# Fabric CI/CD Initial Deployment
Gebruik dit notebook om de repository te clonen en direct naar een Fabric workspace te deployen met `config.yml`.

In [ ]:
%pip install -q fabric-cicd azure-identity

In [ ]:
import os
import tempfile
import subprocess
from pathlib import Path
from fabric_cicd import deploy_with_config

workspace_id = "14cffcef-f00f-4704-85a8-9591654ba168"
environment = "default"
repo_url = "https://github.com/nb-tosch/FabricTest.git"
repo_ref = "main"
item_suffixes = (".Notebook", ".DataPipeline", ".Lakehouse", ".Warehouse")

with tempfile.TemporaryDirectory(prefix="cloned_repo_") as temp_dir:
    print(f"Cloning repo to: {temp_dir}")
    subprocess.run(
        ["git", "clone", "--branch", repo_ref, "--single-branch", repo_url, temp_dir],
        check=True
    )

    repo_root = Path(temp_dir)
    item_dirs = [p for p in repo_root.rglob("*") if p.is_dir() and p.name.endswith(item_suffixes)]
    non_empty_item_dirs = [p for p in item_dirs if any(p.iterdir())]

    print(f"Found {len(item_dirs)} candidate item folders")
    for p in item_dirs:
        print(f" - {p.relative_to(repo_root)}")

    if len(non_empty_item_dirs) == 0:
        raise RuntimeError(
            "No deployable Fabric item folders found. "
            "Expected non-empty folders like <Name>.Notebook / <Name>.DataPipeline / <Name>.Lakehouse / <Name>.Warehouse. "
            "Connect a Fabric workspace to source control and commit the exported artifacts first."
        )

    config_file_path = os.path.join(temp_dir, "config.yml")

    # Override runtime values without changing files in the repo
    config_override = {
        "core": {
            "workspace_id": {environment: workspace_id},
            "repository_directory": "."
        },
        "publish": {"skip": {environment: False}},
        "unpublish": {"skip": {environment: True}}
    }

    result = deploy_with_config(
        config_file_path=config_file_path,
        environment=environment,
        config_override=config_override
    )

    print(f"Deployment status: {result.status}")
    print(f"Deployment message: {result.message}")